In [4]:
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import time
import copy
# import gc

import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import AutoTokenizer, LlamaTokenizer
import joblib

import matplotlib.pyplot as plt
# import seaborn as sns

In [5]:
DEV = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')
layers_to_keep = (0, 1, 23)

# Utils

## Loaders

In [6]:
DEV = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')

def get_opt(model):
    def skip(*args, **kwargs):
        pass
    torch.nn.init.kaiming_uniform_ = skip
    torch.nn.init.uniform_ = skip
    torch.nn.init.normal_ = skip
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(model, torch_dtype='auto')
    model.seqlen = model.config.max_position_embeddings
    return model

def set_seed(seed):
    np.random.seed(seed)
    torch.random.manual_seed(seed)

def get_tokenizer(model):
    if "llama" in model.lower() and "llama-3" not in model.lower():
        tokenizer = LlamaTokenizer.from_pretrained(model, use_fast=False)
        # fix for transformer 4.28.0.dev0 compatibility
        if tokenizer.bos_token_id != 1 or tokenizer.eos_token_id != 2:
            try:
                tokenizer.bos_token_id = 1
                tokenizer.eos_token_id = 2
            except AttributeError:
                pass
    else:
        tokenizer = AutoTokenizer.from_pretrained(model, use_fast=False)
    return tokenizer

def get_wikitext2(nsamples, seed, seqlen, model, tokenizer):

    traindata = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')
    testdata = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')

    trainenc = tokenizer(" ".join(traindata['text']), return_tensors='pt')
    testenc = tokenizer("\n\n".join(testdata['text']), return_tensors='pt')
    print("ts", trainenc.input_ids.shape)

    random.seed(seed)
    trainloader = []
    if False:
        for _ in range(nsamples):
            i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
            j = i + seqlen
            inp = trainenc.input_ids[:, i:j]
            tar = inp.clone()
            tar[:, :-1] = -100
            trainloader.append((inp, tar))
    else:
        i = 0
        for _ in range(nsamples):
#            i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
            j = i + seqlen
            inp = trainenc.input_ids[:, i:j]
            tar = inp.clone()
            tar[:, :-1] = -100
            trainloader.append((inp, tar))
            i = j
    return trainloader, testenc

def get_ptb(nsamples, seed, seqlen, model, tokenizer):
    traindata = load_dataset('ptb_text_only', 'penn_treebank', split='train')
    testdata = load_dataset('ptb_text_only', 'penn_treebank', split='test')

    trainenc = tokenizer(" ".join(traindata['sentence']), return_tensors='pt')
    testenc = tokenizer(" ".join(testdata['sentence']), return_tensors='pt')

    random.seed(seed)
    trainloader = []
    for _ in range(nsamples):
        i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
        j = i + seqlen
        inp = trainenc.input_ids[:, i:j]
        tar = inp.clone()
        tar[:, :-1] = -100
        trainloader.append((inp, tar))
    return trainloader, testenc

def get_c4(nsamples, seed, seqlen, model, tokenizer):
    traindata = load_dataset(
        'allenai/c4', data_files={'train': 'en/c4-train.00000-of-01024.json.gz'}, split='train'
    )
    valdata = load_dataset(
        'allenai/c4', data_files={'validation': 'en/c4-validation.00000-of-00008.json.gz'}, split='validation'
    )

    random.seed(seed)
    trainloader = []
    for _ in range(nsamples):
        while True:
            i = random.randint(0, len(traindata) - 1)
            trainenc = tokenizer(traindata[i]['text'], return_tensors='pt')
            if trainenc.input_ids.shape[1] > seqlen:
                break
        i = random.randint(0, trainenc.input_ids.shape[1] - seqlen - 1)
        j = i + seqlen
        inp = trainenc.input_ids[:, i:j]
        tar = inp.clone()
        tar[:, :-1] = -100
        trainloader.append((inp, tar))

    valenc = tokenizer(' '.join(valdata[:1100]['text']), return_tensors='pt')
    valenc = valenc.input_ids[:, :(256 * seqlen)]

    class TokenizerWrapper:
        def __init__(self, input_ids):
            self.input_ids = input_ids
    valenc = TokenizerWrapper(valenc)

    return trainloader, valenc

def get_loaders(name, nsamples=128, seed=0, seqlen=2048, model=''):
    tokenizer = get_tokenizer(model)
    if 'wikitext2' in name:
        return get_wikitext2(nsamples, seed, seqlen, model, tokenizer)
    if 'ptb' in name:
        return get_ptb(nsamples, seed, seqlen, model, tokenizer)
    if 'c4' in name:
        return get_c4(nsamples, seed, seqlen, model, tokenizer)

## Evaluation

In [7]:
@torch.no_grad()
def eval_ppl(model, testenc, bs=1, device=None):
    # Get input IDs
    testenc = testenc.input_ids

    # Calculate number of samples
    nsamples = testenc.numel() // model.seqlen
    # nsamples = 10
    # List to store negative log likelihoods
    nlls = []

    # Loop through each batch
    for i in range(0,nsamples,bs):
        # Calculate end index
        j = min(i+bs, nsamples)

        # Prepare inputs and move to device
        inputs = testenc[:,(i * model.seqlen):(j * model.seqlen)].to(device)
        inputs = inputs.reshape(j-i, model.seqlen)

        # Forward pass through the model
        lm_logits = model(inputs).logits

        # Shift logits and labels for next token prediction
        shift_logits = lm_logits[:, :-1, :].contiguous()
        shift_labels = inputs[:, 1:]

        # Compute loss
        loss_fct = nn.CrossEntropyLoss()
        loss = loss_fct(shift_logits.reshape(-1, shift_logits.size(-1)), shift_labels.reshape(-1))

        # Calculate negative log likelihood
        neg_log_likelihood = loss.float() * model.seqlen * (j-i)

        # Append to list of negative log likelihoods
        nlls.append(neg_log_likelihood)

    # Compute perplexity
    ppl = torch.exp(torch.stack(nlls).sum() / (nsamples * model.seqlen))

    # Empty CUDA cache to save memory
    torch.cuda.empty_cache()

    return ppl.item()

# Predictor definitions

In [8]:
class LogisticRegression:
    def __init__(self, n_features=2048, lr=0.01, n_iters=1000, patience=10, min_delta=1e-4):
        self.lr = lr
        self.n_iters = n_iters
        self.patience = patience
        self.min_delta = min_delta
        self.model = nn.Sequential(
            nn.Linear(n_features, 1),
            nn.Sigmoid()
        ).to(DEV)
        
    def fit(self, X, y, X_val=None, y_val=None):
        if X_val is None or y_val is None:
            X_val, y_val = X, y
                
        criterion = nn.BCEWithLogitsLoss()  # Binary Cross Entropy Loss
        optimizer = torch.optim.SGD(self.model.parameters(), lr=self.lr)
        
        with torch.no_grad():
            # Xavier/Glorot initialization for the linear layer
            nn.init.xavier_uniform_(self.model[0].weight)
            nn.init.zeros_(self.model[0].bias)
        
        # Early stopping variables
        best_loss = float('inf')
        counter = 0
        best_model_state = None
        
        for i in range(self.n_iters):
            # Forward pass and training loss
            y_pred = self.model(X.to(DEV))
            train_loss = criterion(torch.sigmoid(y_pred), y.to(DEV))
            
            # Backward pass and optimization
            optimizer.zero_grad()
            train_loss.backward()
            optimizer.step()
            
            # Calculate validation loss
            with torch.no_grad():
                val_pred = self.model(X_val.to(DEV))
                val_loss = criterion(val_pred, y_val.to(DEV)).item()
            
            # Check for improvement
            if val_loss < best_loss - self.min_delta:
                best_loss = val_loss
                counter = 0
                # Save the best model
                best_model_state = {k: v.clone() for k, v in self.model.state_dict().items()}
            else:
                counter += 1
            
            # Early stopping check
            if counter >= self.patience:
                break
        
        # Load the best model if we found one
        if best_model_state is not None:
            self.model.load_state_dict(best_model_state)
            
    def predict_proba(self, X):
        return self.model(X.to(DEV)).detach().cpu().numpy()

    def predict(self, X):
        proba = self.predict_proba(X)
        return (proba >= 0.5).astype(int)

In [9]:
hidden_dim = 47
# hidden_dim = 256

class MLPClassifier:
    def __init__(self, n_features=2048, lr=0.01, n_iters=1000, patience=20, min_delta=1e-4):
        self.lr = lr
        self.n_iters = n_iters
        self.patience = patience
        self.min_delta = min_delta
        self.model = nn.Sequential(
            nn.Linear(n_features, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        ).to(DEV)
        
    def fit(self, X, y, X_val=None, y_val=None):        
        if X_val is None or y_val is None:
            X_val, y_val = X, y
        
        criterion = nn.BCEWithLogitsLoss()  # Binary Cross Entropy Loss
        optimizer = torch.optim.SGD(self.model.parameters(), lr=self.lr)
        
        with torch.no_grad():
            nn.init.xavier_uniform_(self.model[0].weight)
            nn.init.zeros_(self.model[0].bias)
            nn.init.xavier_uniform_(self.model[2].weight)
            nn.init.zeros_(self.model[2].bias)
        
        # Early stopping variables
        best_loss = float('inf')
        counter = 0
        best_model_state = None
        
        for i in range(self.n_iters):
            # Forward pass and training loss
            y_pred = self.model(X.to(DEV))
            train_loss = criterion(torch.sigmoid(y_pred), y.to(DEV))
            # if i % 10 == 0: print(train_loss.item(), end=' ')            
            # Backward pass and optimization
            optimizer.zero_grad()
            train_loss.backward()
            optimizer.step()
            
            # Calculate validation loss
            with torch.no_grad():
                val_pred = self.model(X_val.to(DEV))
                val_loss = criterion(val_pred, y_val.to(DEV)).item()
                # if i % 10 == 0: print(val_loss)
            
            # Check for improvement
            if val_loss < best_loss - self.min_delta:
                best_loss = val_loss
                counter = 0
                # Save the best model
                best_model_state = {k: v.clone() for k, v in self.model.state_dict().items()}
            else:
                counter += 1
            
            # Early stopping check
            if counter >= self.patience:
                break
        
        # Load the best model if we found one
        if best_model_state is not None:
            self.model.load_state_dict(best_model_state)
            
    def predict_proba(self, X):
        return self.model(X.to(DEV)).detach().cpu().numpy()

    def predict(self, X):
        proba = self.predict_proba(X.to(DEV))
        return (proba >= 0.5).astype(int)

# Loading

## Model

In [10]:
# model_name = "HuggingFaceTB/SmolLM2-135M"
# model_name = "HuggingFaceTB/SmolLM2-360M"
model_name = "HuggingFaceTB/SmolLM2-1.7B"

model = get_opt(model_name)
model.eval()
model.to(DEV)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 2048)
    (layers): ModuleList(
      (0-23): 24 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb)

## Predictors

In [11]:
for i, layer in enumerate(model.model.layers):
    if i in layers_to_keep: 
        continue
    layer.predictor = {}
    layer.predictor['lr'] = joblib.load(f'predictors/cls_model_layer{i}.pkl')
    layer.predictor['nn'] = joblib.load(f'predictors/cls_model_mlp_layer{i}.pkl')

## Data

In [9]:
n_samples = 32

dataloader, testloader = get_loaders(
    "c4", nsamples=n_samples, seed=0, model=model_name, seqlen=model.seqlen
)

Token indices sequence length is longer than the specified maximum sequence length for this model (8597 > 8192). Running this sequence through the model will result in indexing errors


In [12]:
wiki_dataloader, wiki_testloader = get_loaders(
    'wikitext2', seed=0, model=model_name, seqlen=model.seqlen
)

Token indices sequence length is longer than the specified maximum sequence length for this model (2565857 > 8192). Running this sequence through the model will result in indexing errors


ts torch.Size([1, 2565857])


# Router

In [26]:
class MLPRouter(nn.Module):
    def __init__(self, predictor, mlp):
        super().__init__()
        self.predictor = predictor
        self.mlp = mlp
        self.n_total = 0
        self.n_mlp = 0
        
    def forward(self, x):
        use_mlp = self.predictor.predict(x.to(torch.float)).astype(bool)
        # print(use_mlp.flatten().sum())
        bsz, seq_len, dim = x.shape
        x_flat = x.view(-1, dim)
        use_mlp_flat = use_mlp.flatten()  # shape: (bsz * seq_len)
        self.n_mlp += use_mlp_flat.sum()
        self.n_total += use_mlp_flat.size
        # Apply MLP only to selected tokens
        out = torch.zeros_like(x_flat)  # same shape as input
        if use_mlp_flat.any():
            selected = x_flat[use_mlp_flat]  # (N_selected, dim)
            mlp_out = self.mlp(selected)     # (N_selected, dim)
            out[use_mlp_flat] = mlp_out      # insert results

        return out.view(bsz, seq_len, dim)
        

In [14]:
layer = model.model.layers[5]
predictor = layer.predictor['lr']
mlp_copy = copy.deepcopy(layer.mlp)
layer.mlp = MLPRouter(predictor, mlp_copy)
ppl = eval_ppl(model, wiki_testloader, device=DEV)
ppl

7.7978105545043945

# Evaluate pruned model

In [30]:
def eval_model(use_router='', results_router=None, layer_stats=None):
    if layer_stats is None:
        layer_stats = pd.DataFrame()
    if results_router is None:
        results_router = pd.DataFrame(columns=["perplexity", "inference_time"])

    if use_router:
        predictor_quality = pd.read_csv('all_prediction_results.csv', index_col=0)
        auc_col = 'AUC' if use_router == 'lr' else 'AUC_mlp'
        
        # replace MLP with router only if the AUC is high enough for the layer
        for i, layer in enumerate(model.model.layers):
            if i in layers_to_keep:
                continue
            predictor = layer.predictor[use_router]
            mlp_copy = copy.deepcopy(layer.mlp)
            layer.mlp = MLPRouter(predictor, mlp_copy)
    
    # measure performance
    t0 = time.time()
    ppl = eval_ppl(model, wiki_testloader, device=DEV)
    t = time.time() - t0
    results_router.loc[use_router] = [ppl, t]

    # repair
    if use_router:
        sum_n_total = sum_n_skipped = 0
        column_name = use_router

        for i, layer in enumerate(model.model.layers):
            if i in layers_to_keep:
                continue

            n_total = layer.mlp.n_total
            n_skipped = n_total - layer.mlp.n_mlp
            skip_ratio = n_skipped / n_total if n_total > 0 else 0

            sum_n_skipped += n_skipped

            layer_stats.at[i, column_name] = skip_ratio
            layer.mlp = copy.deepcopy(layer.mlp.mlp)  # restore

        sum_n_total = n_total * len(model.model.layers)
        global_skip_ratio = sum_n_skipped / sum_n_total
        layer_stats.at['all', column_name] = global_skip_ratio

    return results_router, layer_stats

In [31]:
results_router = pd.DataFrame(columns=["perplexity", "inference_time"])

results_router, layer_stats = eval_model('lr', results_router=results_router)
results_router, layer_stats = eval_model('nn', results_router=results_router, layer_stats=layer_stats)
results_router, layer_stats = eval_model('', results_router=results_router, layer_stats=layer_stats)
results_router.to_csv('results_router.csv')
results_router

,perplexity,inference_time
lr,11.536591,9.705074
nn,1058.414307,8.444103
,7.260689,9.250544


In [44]:
# stare vysledky
results_router = pd.DataFrame(columns=["perplexity", "inference_time"])

results_router, layer_stats = eval_model('lr', results_router=results_router)
results_router, layer_stats = eval_model('nn', results_router=results_router, layer_stats=layer_stats)
results_router, layer_stats = eval_model('', results_router=results_router, layer_stats=layer_stats)
results_router.to_csv('results_router.csv')
results_router

,perplexity,inference_time
lr,1.139075e+07,9.462785
nn,5.924430e+06,9.524502
,7.260689e+00,9.412255


## Ratio of tokens routed around the MLP

In [32]:
layer_stats

,lr,nn
2,0.356693,0.446197
3,0.270145,0.438460
4,0.172957,0.407382
5,0.000102,0.771712
6,0.000000,0.414026
7,0.012461,0.548633
8,0.000003,0.367237
9,0.012883,0.331992
10,0.000000,0.086766
11,0.000000,0.038238


# Using only predictors with higher AUC

In [33]:
def eval_model_with_AUC_threshold(use_router='', results_router=None, layer_stats=None, auc_thr=0.75):
    if layer_stats is None:
        layer_stats = pd.DataFrame()
    if results_router is None:
        results_router = pd.DataFrame(columns=["perplexity", "inference_time"])

    if use_router:
        predictor_quality = pd.read_csv('all_prediction_results.csv', index_col=0)
        auc_col = 'AUC' if use_router == 'lr' else 'AUC_mlp'
        
        # replace MLP with router only if the AUC is high enough for the layer
        for i, layer in enumerate(model.model.layers):
            if i in layers_to_keep or predictor_quality.loc[i, auc_col] < auc_thr:
                continue
            predictor = layer.predictor[use_router]
            mlp_copy = copy.deepcopy(layer.mlp)
            layer.mlp = MLPRouter(predictor, mlp_copy)
    
    # measure performance
    t0 = time.time()
    ppl = eval_ppl(model, wiki_testloader, device=DEV)
    t = time.time() - t0
    results_router.loc[use_router] = [ppl, t]

    # repair
    if use_router:
        sum_n_total = sum_n_skipped = 0
        column_name = use_router

        for i, layer in enumerate(model.model.layers):
            if i in layers_to_keep or predictor_quality.loc[i, auc_col] < auc_thr:
                continue

            n_total = layer.mlp.n_total
            n_skipped = n_total - layer.mlp.n_mlp
            skip_ratio = n_skipped / n_total if n_total > 0 else 0

            sum_n_skipped += n_skipped

            layer_stats.at[i, column_name] = skip_ratio
            layer.mlp = copy.deepcopy(layer.mlp.mlp)  # restore

        sum_n_total = n_total * len(model.model.layers)
        global_skip_ratio = sum_n_skipped / sum_n_total
        layer_stats.at['all', column_name] = global_skip_ratio

    return results_router, layer_stats

In [34]:
results_router_thr = pd.DataFrame(columns=["perplexity", "inference_time"])

results_router_thr, layer_stats_thr = eval_model_with_AUC_threshold('lr', results_router=results_router_thr, auc_thr=0.8)
results_router_thr, layer_stats_thr = eval_model_with_AUC_threshold('nn', results_router=results_router_thr, layer_stats=layer_stats_thr, auc_thr=0.8)
results_router_thr, layer_stats_thr = eval_model_with_AUC_threshold('', results_router=results_router_thr, layer_stats=layer_stats_thr, auc_thr=0.8)
results_router_thr.to_csv('results_router_thr.csv')
results_router_thr

,perplexity,inference_time
lr,9.461728,9.022177
nn,15.059263,8.684023
,7.260689,9.305189


In [39]:
# stare vysledky
results_router_thr = pd.DataFrame(columns=["perplexity", "inference_time"])

results_router_thr, layer_stats_thr = eval_model_with_AUC_threshold('lr', results_router=results_router_thr, auc_thr=0.8)
results_router_thr, layer_stats_thr = eval_model_with_AUC_threshold('nn', results_router=results_router_thr, layer_stats=layer_stats_thr, auc_thr=0.8)
results_router_thr, layer_stats_thr = eval_model_with_AUC_threshold('', results_router=results_router_thr, layer_stats=layer_stats_thr, auc_thr=0.8)
results_router_thr.to_csv('results_router_thr.csv')
results_router_thr

,perplexity,inference_time
lr,178.115570,9.344595
nn,427.999298,9.386919
,7.260689,9.345299


In [35]:
layer_stats_thr

,lr,nn
2,0.356693,0.446197
3,0.270145,0.438460
4,0.172957,0.407382
19,0.438354,0.621638
20,0.097033,0.396837
21,0.190344,0.776684
22,0.817234,0.828415
all,0.097615,0.195305
5,NaN,0.771712


---
# Top-k router

In [13]:
class MLPRouterTopK(nn.Module):
    def __init__(self, predictor, mlp, k=0.7):
        super().__init__()
        self.predictor = predictor
        self.mlp = mlp
        self.k = k
        self.n_total = 0
        self.n_mlp = 0
        
    def forward(self, x):
        # compute logits
        use_mlp_probs = torch.Tensor(self.predictor.predict_proba(x.to(torch.float)))
        bsz, seq_len, dim = x.shape
        x_flat = x.view(-1, dim)

        k_per_row = max(1, int(seq_len * self.k))  # ensure at least 1
        selected_idx = torch.topk(use_mlp_probs, k_per_row, dim=1).indices.flatten()  # (bsz, k_per_row)
        
        self.n_mlp += selected_idx.numel()
        self.n_total += use_mlp_probs.numel()
        # Apply MLP only to selected tokens
        out = torch.zeros_like(x_flat)  # same shape as input
        if selected_idx.numel():
            selected = x_flat[selected_idx]  # (N_selected, dim)
            mlp_out = self.mlp(selected)     # (N_selected, dim)
            out[selected_idx] = mlp_out      # insert results

        return out.view(bsz, seq_len, dim)
        

In [14]:
def eval_model_topk(use_router='', results_router=None, layer_stats=None, k=0.7):
    if layer_stats is None:
        layer_stats = pd.DataFrame()
    if results_router is None:
        results_router = pd.DataFrame(columns=["perplexity", "inference_time"])

    if use_router:
        predictor_quality = pd.read_csv('all_prediction_results.csv', index_col=0)
        auc_col = 'AUC' if use_router == 'lr' else 'AUC_mlp'
        
        # replace MLP with router only if the AUC is high enough for the layer
        for i, layer in enumerate(model.model.layers):
            if i in layers_to_keep:
                continue
            predictor = layer.predictor[use_router]
            mlp_copy = copy.deepcopy(layer.mlp)
            layer.mlp = MLPRouterTopK(predictor, mlp_copy, k=k)
    
    # measure performance
    t0 = time.time()
    ppl = eval_ppl(model, wiki_testloader, device=DEV)
    t = time.time() - t0
    results_router.loc[use_router] = [ppl, t]

    # repair
    if use_router:
        sum_n_total = sum_n_skipped = 0
        column_name = use_router

        for i, layer in enumerate(model.model.layers):
            if i in layers_to_keep:
                continue

            n_total = layer.mlp.n_total
            n_skipped = n_total - layer.mlp.n_mlp
            skip_ratio = n_skipped / n_total if n_total > 0 else 0

            sum_n_skipped += n_skipped

            layer_stats.at[i, column_name] = skip_ratio
            layer.mlp = copy.deepcopy(layer.mlp.mlp)  # restore

        sum_n_total = n_total * len(model.model.layers)
        global_skip_ratio = sum_n_skipped / sum_n_total
        layer_stats.at['all', column_name] = global_skip_ratio

    return results_router, layer_stats

In [18]:
results_router_top70 = pd.DataFrame(columns=["perplexity", "inference_time"])

results_router_top70, layer_stats_top70 = eval_model_topk('lr', results_router=results_router_top70)
results_router_top70, layer_stats_top70 = eval_model_topk('nn', results_router=results_router_top70, layer_stats=layer_stats_top70)
results_router_top70, layer_stats_top70 = eval_model_topk('', results_router=results_router_top70, layer_stats=layer_stats_top70)
results_router_top70.to_csv('results_router_top70.csv')
results_router_top70

,perplexity,inference_time
lr,169.028595,8.997414
nn,252.139633,9.041593
,7.260689,9.297124


In [19]:
layer_stats_top70

,lr,nn
2,0.300049,0.300049
3,0.300049,0.300049
4,0.300049,0.300049
5,0.300049,0.300049
6,0.300049,0.300049
7,0.300049,0.300049
8,0.300049,0.300049
9,0.300049,0.300049
10,0.300049,0.300049
11,0.300049,0.300049


In [16]:
results_router_top80 = pd.DataFrame(columns=["perplexity", "inference_time"])

results_router_top80, layer_stats_top80 = eval_model_topk('lr', results_router=results_router_top80, k=0.8)
results_router_top80, layer_stats_top80 = eval_model_topk('nn', results_router=results_router_top80, layer_stats=layer_stats_top80, k=0.8)
results_router_top80, layer_stats_top80 = eval_model_topk('', results_router=results_router_top80, layer_stats=layer_stats_top80, k=0.8)
results_router_top80.to_csv('results_router_top80.csv')
results_router_top80

,perplexity,inference_time
lr,64.782173,13.072218
nn,66.248466,9.408454
,7.260689,9.242539


In [17]:
layer_stats_top80

,lr,nn
2,0.200073,0.200073
3,0.200073,0.200073
4,0.200073,0.200073
5,0.200073,0.200073
6,0.200073,0.200073
7,0.200073,0.200073
8,0.200073,0.200073
9,0.200073,0.200073
10,0.200073,0.200073
11,0.200073,0.200073
